In [ ]:
# def calibrate_equity_spot_local_volatility_model(context: Any,
#                                                  results: CalibratorResult,
#                                                  mkt_data_cache: Any) -> None:
#     """
#     Main driver [] iterates over equities, builds spot and advance [] swap
#     series, calibrates the Heston parameters and writes them back onto the
#     equity objects.
#     """

#     # 1 Compute (once) the 1-day log-returns that will be used later
#     filled_log_returns = calc_and_fill_log_returns_from_underlyings(
#         context, results, mkt_data_cache
#     )

#     # 2 Loop over every equity in the results scope
#     for equity in results.get_children():

#         raw_spot_series = results.raw_data_dls.Datalines.get(equity.name)

#         if raw_spot_series is None:
#             # No market data - keep the defaults (0.0,0.0,0.0)
#             equity.kappa = equity.theta = equity.eta = equity.rho = 0.0
#             continue

#     spot_ts = fill_levels_from_log_returns(
#         raw_spot_series,
#         filled_log_returns["Price"],
#         results.data_dates,
#         results.data,
#     )

#     # Variance-swap series for each tenor
#     var_swap_ts: Dict[int, List[DataPoint]] = {}
#     raw_vs_returns: Dict[int, List[float]] = {}

#     for tenor in vs_varSwapTenors:
#         var_id = f"VarSwap_{equity.name}_{tenor}"
#         raw_vs = results.raw_data_dls.DataLines.get(var_id)
#         if raw_vs is None:
#             continue

#         vs_ts = fill_levels_from_log_returns(
#             raw_vs,
#             filled_log_returns["VarSwap"],
#             results.data_dates,
#             results.data,
#         )


#         vs_ret = ReturnsGenerator.returns(
#             vs_ts,
#             returns_type="absolute",
#             lags=results.schedule.NDayReturns,
#         )

#         var_swap_ts[tenor] = vs_ts
#         var_swap_returns[tenor] = vs_ret

#         heston_params = get_heston_model_parameters(
#             spot_ts,
#             var_swap_ts,
#             var_swap_returns,
#             results.data_dates,
#             results.schedule.NDayReturns,
#             equity.vol,
#         )

#         # simple sanity check (identical to the C# version)
#         kappa, _, _, lam = heston_params
#         valid = (0.0 < kappa < 10.0) and (lam >= 0.0)

#         equity.kappa = heston_params[0] if valid else 0.0
#         equity.theta = heston_params[1] if valid else 0.0
#         equity.rho = heston_params[2] if valid else 0.0
#         equity.eta = heston_params[3] if valid else 0.0

# def get_heston_model_parameters(equity_spot_ts,
#                                 inst_var_ts,
#                                 equity_var_swap_ts,
#                                 interest_var_swap_ts) -> tuple[float, float, float, float, float]:
    
#     kappa = calibrate_heston_model_kappa(equity_var_swap_ts, inst_var_ts)
#     theta = volatility * volatility

#     inst_var_ts = get_instantaneous_variance_heston(
#         equity_var_swap_ts,
#         equity_spot_ts,
#         interest_var_swap_ts
#     )

#     inst_var_ret = ReturnsGenerator.get_returns(
#         inst_var_ts,
#         returns_type="absolute", lag=delta_t
#     )

#     eta, d_z = calibrate_heston_model_eta(
#         inst_var_ts, inst_var_ret, kappa, theta, delta_t
#     )

#     rho = calibrate_heston_model_rho(equity_spot_ts, inst_var_ts, d_z, delta_t)

#     lam = kappa * 0.5 * rho * eta
#     return kappa, theta, eta, rho, lam

# def calibrate_heston_model_kappa(equity_var_swap_returns: Dict[int, List[float]]) -> float:
#     """OLS regression of short-tenor vs long-tenor variance swap returns."""
#     y = equity_var_swap_returns.get(s_varSwapTenors[1], [])
#     x = equity_var_swap_returns.get(s_varSwapTenors[0], [])

#     # Remove NaNs before regression
#     x_clean = [v for v in x if not math.isnan(v)]
#     y_clean = [v for v in y if not math.isnan(v)]

#     intercept, slope = LinearLeastSquares.straight_line_regression(x_clean, y_clean)

#     short = s_varSwapTenors[0] / 12.0  # # months -> years
#     long = s_varSwapTenors[1] / 12.0

#     kappa = (2.0 * (1.0 - slope)) / (long - short * slope)
#     return kappa

# def get_instantaneous_variance_heston(equity_var_swap_ts: Dict[int, List[DataPoint]],
#                                        kappa: float,
#                                        theta: float,
#                                        factor: float,
#                                        dates: List[datetime]) -> List[DataPoint]:
#     """Maps a short[tenor variance swap price to an instantaneous variance."""
#     short_ts = equity_var_swap_ts.get(s_varSwapTenors[0])
#     if short_ts is None:
#         raise ValueError("Missing short[tenor variance swap series")

#     tau = s_varSwapTenors[0] / 12.0
#     factor = (kappa * tau) / (1.0 - math.exp(-kappa * tau))

#     transformed = [
#         DataPoint(
#             dp.date,
#             dp.value * theta + factor * theta
#         )
#         for dp in short_ts
#     ]

#     # Clip very small negative values to zero (C# used 1e-6)
#     clean_vals = [v if v > 1e-6 else 0.0 for v in DataPointHelper.to_values(transformed)]

#     return DataPointHelper.create_data_points(dates, clean_vals)

# def calibrate_heston_model_eta(instantaneous_variance_ts: List[DataPoint],
#                                 instantaneous_variance_returns: List[DataPoint],
#                                 kappa: float,
#                                 theta: float,
#                                 delta: int) -> Tuple[float, List[float]]:
#     """Derives η (vol-of-vol) and the normalised residual series of eta."""
#     delta_years = delta / DaysPerYear

#     sqrt_v = [math.sqrt(v) for v in DataPointHelper.to_values(instantaneous_variance_ts)]

#     eta_dz = []
#     for i, r in enumerate(instantaneous_variance_returns):
#         v_term = kappa * (theta - instantaneous_variance_ts[i].value)
#         sqrt_v_term = (sqrt_v[i + 1] - sqrt_v[i]) * delta_years
#         dz = (r.value - v_term) / sqrt_v_term
#         if math.isnan(dz) or abs(dz) > 5:
#             dz = math.nan
#         eta_dz.append(dz)

#     # Remove infinities / NaNs caused by division-by-zero
#     eta_dz_clean = [x for x in eta_dz if not math.isnan(x) else math.nan for x in eta_dz]
#     σ = stdev([x for x in eta_dz_clean if not math.isnan(x)])
#     eta = Descriptive.stdev(scaled)

#     # Normalise to obtain dz = (η dz)/σ
#     eta_dz_clean = [(x * eta) / σ if not math.isnan(x) for x in eta_dz_clean]

#     return eta, eta_dz

# def calibrate_heston_model_rho(equity_spot_ts: List[DataPoint],
#                                 instantaneous_variance_ts: List[DataPoint],
#                                 d_z: List[float],
#                                 delta_t: int) -> float:
#     """
#     Correlation between spot returns and the variance Process Brownian motion.
#     """
#     sqrt_v = [math.sqrt(v) for v in DataPointHelper.to_values(instantaneous_variance_ts)]

#     d_w = ReturnsGenerator.returns(
#         equity_spot_ts,
#         returns_type="relative_simple",
#         lag=delta_t,
#     )

#     d_z = [
#         r / sv if sv != 0 else math.nan
#         for r, sv in zip(spot_ret, sqrt_v)
#     ]

#     rho = Descriptive.correlation(d_z, d_w)
#     if math.isnan(rho):
#         rho = 0.0
#     # Guard against tiny rounding errors that could push the value >1
#     return min(rho, 1.0)



In [1]:
from __future__ import annotations
import math
from typing import Tuple
import numpy as np
import pandas as pd
from scipy import stats


DaysPerYear = 252  # trading days convention


def _safe_std(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    return float(np.std(x, ddof=1)) if x.size > 1 else 0.0


def straight_line_regression(x: np.ndarray, y: np.ndarray) -> Tuple[float, float]:
    """Return intercept, slope using OLS; safe for short arrays."""
    mask = (~np.isnan(x)) & (~np.isnan(y))
    if mask.sum() < 2:
        return 0.0, 1.0
    slope, intercept, _, _, _ = stats.linregress(x[mask], y[mask])
    # stats.linregress returns slope, intercept order (slope, intercept)
    # We return intercept, slope to match your original usage.
    return float(intercept), float(slope)


def calibrate_kappa_from_varswap_returns(varswap_returns: dict,
                                         short_tenor_months: int,
                                         long_tenor_months: int) -> float:
    """
    Recreates the OLS short-vs-long regression and returns kappa using:
    kappa = (2 * (1 - slope)) / (long - short * slope)
    where long/short are in years.
    """
    x = np.array(varswap_returns.get(short_tenor_months, []), dtype=float)
    y = np.array(varswap_returns.get(long_tenor_months, []), dtype=float)

    # drop nan-paired observations
    mask = (~np.isnan(x)) & (~np.isnan(y))
    if mask.sum() < 2:
        return 1.0  # fallback

    intercept, slope = straight_line_regression(x[mask], y[mask])

    short = short_tenor_months / 12.0
    long = long_tenor_months / 12.0

    denom = (long - short * slope)
    if abs(denom) < 1e-12:
        return 1.0
    kappa = (2.0 * (1.0 - slope)) / denom
    return float(kappa)


def instantaneous_variance_from_short_varswap(short_varswap: pd.Series,
                                              kappa: float,
                                              theta: float,
                                              short_tenor_months: int) -> pd.Series:
    """
    Map short-tenor variance swap series to an instantaneous variance series.
    Uses the same algebraic form you had:
      tau = short_tenor_months / 12
      factor = (kappa * tau) / (1 - exp(-kappa * tau))
      inst_var = short_varswap * theta + factor * theta
    Input/Output as pandas Series indexed by dates.
    """
    if short_varswap.empty:
        raise ValueError("short_varswap series is empty")

    tau = short_tenor_months / 12.0
    denom = 1.0 - math.exp(-kappa * tau)
    factor = (kappa * tau) / denom if denom != 0 else 1.0

    inst_vals = (short_varswap.values.astype(float) * theta) + (factor * theta)
    inst_vals = np.where(inst_vals > 1e-12, inst_vals, 0.0)  # clip small/negative to 0
    return pd.Series(inst_vals, index=short_varswap.index).astype(float)


def compute_returns(series: pd.Series, returns_type: str = "absolute", lag: int = 1) -> pd.Series:
    """
    simple wrapper to compute returns.
    returns_type:
      - 'absolute' : x_t - x_{t-lag}
      - 'relative_simple' : x_t / x_{t-lag} - 1
    lag in number of observations (e.g., days)
    """
    if returns_type == "absolute":
        return series.diff(periods=lag)
    elif returns_type == "relative_simple":
        return series.pct_change(periods=lag)
    else:
        raise ValueError("unsupported returns_type")


def calibrate_eta(instantaneous_variance: pd.Series,
                  inst_var_returns: pd.Series,
                  kappa: float,
                  theta: float,
                  delta_days: int) -> Tuple[float, np.ndarray]:
    """
    Derive vol-of-vol (eta) and normalized residuals (eta * dz or dz depending).
    Methodology mirrors your original code:

    sqrt_v = sqrt(v)
    for each t:
      v_term = kappa*(theta - v_t)
      sqrt_v_term = (sqrt_v[t+1] - sqrt_v[t]) * delta_years
      dz = (r_t - v_term) / sqrt_v_term

    Then compute standard deviation of dz (σ) and set eta = σ.
    Returns: eta (float), normalized_dz (array with same length-1 as returns)
    """
    v = instantaneous_variance.values.astype(float)
    r = inst_var_returns.values.astype(float)

    delta_years = delta_days / float(DaysPerYear)
    n = len(v)
    if n < 2:
        return 0.0, np.array([])

    sqrt_v = np.sqrt(np.maximum(v, 0.0))
    dz = np.full(n - 1, np.nan, dtype=float)

    for i in range(n - 1):
        v_term = kappa * (theta - v[i])
        sqrt_v_term = (sqrt_v[i + 1] - sqrt_v[i]) * delta_years
        # guard division by zero / tiny denom
        if abs(sqrt_v_term) < 1e-12:
            dz[i] = np.nan
            continue
        dz_val = (r[i] - v_term) / sqrt_v_term
        if math.isnan(dz_val) or abs(dz_val) > 1e6:
            dz[i] = np.nan
        else:
            dz[i] = dz_val

    sigma = _safe_std(dz)
    eta = float(sigma)  # this follows your code where eta comes from std of dz
    # Normalise residuals to obtain "dz_normalized = dz / sigma" (guard)
    normalized = dz.copy()
    if sigma > 0:
        normalized = np.array([x / sigma if not math.isnan(x) else np.nan for x in dz])
    else:
        normalized = np.array([np.nan for _ in dz])

    return eta, normalized


def calibrate_rho(spot_series: pd.Series,
                  instantaneous_variance: pd.Series,
                  normalized_dz: np.ndarray,
                  delta_days: int) -> float:
    """
    Correlation between spot returns (dW) and the variance-process Brownian motion (dZ).
    We compute spot simple returns with lag delta_days and correlate with normalized_dz aligned.
    """
    # compute spot simple returns aligned with variance increments
    d_w = compute_returns(spot_series, returns_type="relative_simple", lag=delta_days).values
    # d_w length equals len(spot_series) - delta_days; normalized_dz length is len(instantaneous_variance)-1
    # Align lengths by trimming to the minimum overlapping length from the beginning
    min_len = min(len(d_w), len(normalized_dz))
    if min_len <= 0:
        return 0.0

    arr_w = d_w[-min_len:]
    arr_z = normalized_dz[-min_len:]
    mask = (~np.isnan(arr_w)) & (~np.isnan(arr_z))
    if mask.sum() < 2:
        return 0.0
    rho = float(np.corrcoef(arr_w[mask], arr_z[mask])[0, 1])
    if math.isnan(rho):
        return 0.0
    # guard tiny numerical >1
    rho = max(min(rho, 1.0), -1.0)
    return rho


def get_heston_parameters_from_varswaps(spot_series: pd.Series,
                                        short_varswap: pd.Series,
                                        long_varswap: pd.Series,
                                        varswap_returns_by_tenor: dict,
                                        short_tenor_months: int,
                                        long_tenor_months: int,
                                        delta_days: int) -> dict:
    """
    Main pipeline that returns a dictionary with calibrated Heston parameters:
      kappa, theta, eta, rho, lam
    Inputs:
      - spot_series: pd.Series indexed by date, spot prices
      - short_varswap: pd.Series indexed by date, short-tenor variance swap prices
      - long_varswap: pd.Series indexed by date, long-tenor variance swap prices
      - varswap_returns_by_tenor: dict{tenor_months: list/array of returns} used for kappa regression
      - short_tenor_months, long_tenor_months: ints (e.g., 1, 12)
      - delta_days: int (lag in days for returns)
    Notes/assumptions:
      - long_varswap is used to estimate theta (annual variance level)
      - short_varswap is used to derive instantaneous variance (per your formula)
    """
    # 1) Calibrate kappa from returns regression
    kappa = calibrate_kappa_from_varswap_returns(varswap_returns_by_tenor,
                                                 short_tenor_months,
                                                 long_tenor_months)

    # 2) Estimate theta: use average long-tenor variance swap (annual variance estimate)
    theta = float(np.nanmean(long_varswap.values)) if not long_varswap.empty else 0.0

    # 3) Instantaneous variance series from short tenor
    inst_var = instantaneous_variance_from_short_varswap(short_varswap, kappa, theta, short_tenor_months)

    # 4) Compute returns for instantaneous variance (absolute returns as in your original code)
    inst_var_returns = compute_returns(inst_var, returns_type="absolute", lag=delta_days).dropna()
    # align inst_var to returns (we need equal-length arrays)
    inst_var_aligned = inst_var.loc[inst_var_returns.index]

    # 5) Calibrate eta and get normalized dz residuals
    eta, normalized_dz = calibrate_eta(inst_var_aligned, inst_var_returns, kappa, theta, delta_days)

    # 6) Calibrate rho between spot returns and var-process Brownian increments
    rho = calibrate_rho(spot_series.reindex(inst_var_aligned.index, method="ffill"),
                        inst_var_aligned,
                        normalized_dz,
                        delta_days)

    # 7) market price of volatility lambda approximation from your formula
    lam = float(kappa * 0.5 * rho * eta)

    return {
        "kappa": float(kappa),
        "theta": float(theta),
        "eta": float(eta),
        "rho": float(rho),
        "lambda": float(lam),
        "inst_variance_series": inst_var  # include for diagnostics
    }


# If you want a helper to compute variance-swap fair strike from options, here's
# a simple discretized Carr-Madan style estimator. The user must supply an options
# DataFrame with strikes and call/put mid prices and an at-the-money forward.
def variance_swap_from_options(options_df: pd.DataFrame,
                               fwd: float,
                               dt: float) -> float:
    """
    options_df must contain 'strike' and 'mid' (option mid price) and a column 'type' in {'call','put'}.
    This is a discrete approximation of the integral used to compute fair variance swap strike:
      K_var ≈ (2/T) * sum_{i} (ΔK_i / K_i^2) * option_price(K_i)  - (1/T)*(F/K0 - 1)^2
    This is educational — for production use refine interpolation, spacing, and continuity across ATM.
    dt = time-to-expiry in years
    """
    # Ensure sorted by strike
    df = options_df.sort_values("strike").copy()
    strikes = df["strike"].values
    prices = df["mid"].values

    # compute ΔK as midpoints between strikes
    dK = np.empty_like(strikes)
    dK[1:-1] = (strikes[2:] - strikes[:-2]) / 2.0
    dK[0] = strikes[1] - strikes[0]
    dK[-1] = strikes[-1] - strikes[-2]

    integrand = (dK / (strikes ** 2)) * prices
    integral = np.nansum(integrand)
    var_strike = (2.0 / dt) * integral - (1.0 / dt) * ((fwd / strikes[np.argmin(np.abs(strikes - fwd))]) - 1.0) ** 2
    return float(max(var_strike, 0.0))


if __name__ == "__main__":
    # Example usage (replace with your actual series/data)
    # -------------------------
    # Create example (toy) series:
    dates = pd.date_range("2024-01-01", periods=200, freq="B")
    # toy spot
    spot = pd.Series(100 + np.cumsum(np.random.normal(0, 0.5, size=len(dates))), index=dates)
    # toy short and long varswap prices (annual variance units)
    short_var = pd.Series(0.04 + 0.005 * np.random.normal(size=len(dates)), index=dates)
    long_var = pd.Series(0.05 + 0.005 * np.random.normal(size=len(dates)), index=dates)

    # Example returns-by-tenor used in kappa regression: lists of returns for short and long
    varswap_returns_by_tenor = {
        1: list(np.random.normal(0.0, 0.001, size=120)),    # short tenor (months)
        12: list(np.random.normal(0.0, 0.001, size=120)),   # long tenor (months)
    }

    params = get_heston_parameters_from_varswaps(
        spot_series=spot,
        short_varswap=short_var,
        long_varswap=long_var,
        varswap_returns_by_tenor=varswap_returns_by_tenor,
        short_tenor_months=1,
        long_tenor_months=12,
        delta_days=1
    )

    print("Calibrated Heston params:")
    for k, v in params.items():
        if k != "inst_variance_series":
            print(f"  {k}: {v}")


C:\Users\akhil\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Calibrated Heston params:
  kappa: 2.0366363417829283
  theta: 0.04961232409448831
  eta: 52558.464935148026
  rho: 0.05519169531063805
  lambda: 2953.9279639733445
